# Image Classification Baseline (Keras)

Generated by `flashkeras.notebooks`.

Assumes your images are organized as:
```
data_dir/
    class_a/
        img1.jpg
        img2.jpg
    class_b/
        img3.jpg
```
Edit the **Parameters** cell below, then `Run All`.

In [ ]:
data_dir = "REPLACE_ME/path/to/data_dir"
image_size = (128, 128)
batch_size = 32
validation_split = 0.2
epochs = 15
seed = 42

## 1. Load data + augmentation

In [ ]:
from flashkeras.data_collecting.images import load_all_classes_from_directory_and_preprocess_test_split

train_ds, val_ds = load_all_classes_from_directory_and_preprocess_test_split(
    data_dir,
    img_shape=image_size,
    color_mode='grayscale',
    horizontal_flip=True,
    rotation_range=20,
    batch_size=batch_size
)

class_names = train_ds.class_indices
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")


## 2. Peek at some samples

In [ ]:
from flashkeras.analysing.images import show_images_from_batch

show_images_from_batch(train_ds, num_images=3)

## 3. Build baseline CNN

In [ ]:
from flashkeras.models.layers import *
from flashkeras.models import FlashSequential

flash = FlashSequential('classification')

flash.addFlatten()
flash.add(layers.Dense(64, activation="relu")) # It is also compatible with keras!
flash.addDense(32, 'elu')


In [ ]:
print("Input shape:", flash.model.inputs)
print("Shape das imagens:", train_ds.image_shape)
print("Batch shape:", train_ds[0][0].shape)

## 4. Train

In [ ]:
from flashkeras.utils.kerasimports import keras

history = flash.train(x=train_ds, epochs=15, validation_data=val_ds, auto_output_layer=True)


## 5. Training curves

In [ ]:
from flashkeras.analysing.graphs.evaluation import plot_history_train_curve

plot_history_train_curve(history)

## 6. Evaluate

In [ ]:
from flashkeras.evaluation import getAccuracy, getRecall

acc = getAccuracy(flash, val_ds)
print(f"Acc: {acc}")
recall = getRecall(flash, val_ds)
print(f"Recall: {recall}")

## Next steps

- Try transfer learning (e.g. `keras.applications.MobileNetV2`) instead of training from scratch
- Tune `image_size`, `batch_size`, and augmentation strength
- Add a confusion matrix / per-class metrics for deeper error analysis
- Save the model with `model.save("model.keras")`